# Crypto Strategy Run And Validation Walkthrough

This notebook uses `untested/crypto_perp_autoresearch_ensemble.py` as the example strategy.

It shows three things:

1. how to run the strategy in runner `screen` mode;
2. how to run the same config in runner `validate` mode;
3. how to read the important artifacts and sanity-check that the output is realistic from a quant-trading perspective.

The validation here is still **runner smoke validation**, not promotion evidence, market robustness, or permission to move the strategy to `tested/`.

## Setup

Run this notebook from the repository or from `notebooks/`. It finds the repo root, adds the local package paths, and imports the runner API directly.

In [1]:
from __future__ import annotations

import csv
import json
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "pyproject.toml").exists() and (path / "runs").exists():
            return path
    raise RuntimeError("Could not find quant_strategies repository root")


REPO_ROOT = find_repo_root()
for path in (REPO_ROOT, REPO_ROOT / "src"):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from quant_strategies.runner import run_config

SCREEN_CONFIG = REPO_ROOT / "runs" / "crypto_perp_autoresearch_ensemble_smoke.toml"
assert SCREEN_CONFIG.exists(), SCREEN_CONFIG

REPO_ROOT

PosixPath('/Users/Season_Yang/Personal/quant_strategies')

## Inspect The Config

This is the committed smoke config. It points at the untested strategy, loads `crypto_perp_funding` data for BTC/ETH/SOL, and runs in `screen` mode.

In [2]:
print(SCREEN_CONFIG.read_text())

strategy_path = "untested/crypto_perp_autoresearch_ensemble.py"
strategy_id = "crypto_perp_autoresearch_ensemble_smoke"

[data]
kind = "crypto_perp_funding"
symbols = [
  "BTC-PERP",
  "ETH-PERP",
  "SOL-PERP",
]
start = "2024-01-01"
end = "2024-01-07"
strict = true

[params]
symbols = [
  "BTC-PERP",
  "ETH-PERP",
  "SOL-PERP",
]
min_votes = 4
base_position_pct = 0.08
cooldown_bars = 2
decision_interval_minutes = 60
decision_lag_minutes = 1
max_hold_bars = 720
atr_stop_mult = 5.5
rsi_period = 8
bb_percentile_threshold = 85.0
momentum_long_hours = 12
momentum_short_hours = 6
vshort_threshold_mult = 0.5
vol_lookback_hours = 48
base_momentum_threshold = 0.012
target_volatility = 0.015
dynamic_threshold_floor = 0.006
dynamic_threshold_ceiling = 0.025
ema_fast_span = 12
ema_slow_span = 26
macd_fast_span = 12
macd_slow_span = 26
macd_signal_span = 9
atr_period = 24
bb_window_hours = 10
bb_percentile_window_hours = 20

[fill_model]
price = "close"
entry_lag_bars = 1
exit_lag_bars = 0

[cost_

## Run: Screen Mode

`screen` mode asks the strategy for decisions, adapts decisions to engine signals, and computes trade evidence. It does **not** apply validation gates.

Equivalent CLI command:

```bash
conda run -n quant quant-strategies run runs/crypto_perp_autoresearch_ensemble_smoke.toml
```

In [3]:
screen_result = run_config(SCREEN_CONFIG, repo_root=REPO_ROOT)
assert screen_result.success, screen_result.message
screen_result.result_dir

PosixPath('/Users/Season_Yang/Personal/quant_strategies/results/2026-05-27T125032Z-crypto_perp_autoresearch_ensemble_smoke')

In [4]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text())


def parse_ts(value: str) -> datetime:
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


screen_summary = read_json(screen_result.result_dir / "summary.json")
screen_summary["engine"]

{'passed': None,
 'smoke_score': {'sum_weighted_trade_cost_return': 0.0,
  'sum_weighted_trade_funding_return': 3.599333333333333e-05,
  'sum_weighted_trade_gross_return': 0.0033154196420537695,
  'sum_weighted_trade_net_return': 0.003351412975387103},
 'trade_count': 26}

### How To Read The Screen Output

- `trade_count`: number of independent trades the smoke engine scored.
- `sum_weighted_trade_gross_return`: price return times target weight, summed over trades.
- `sum_weighted_trade_funding_return`: funding PnL contribution for supplied funding events during held intervals.
- `sum_weighted_trade_cost_return`: configured fee/slippage cost contribution. The smoke config uses zero costs.
- `sum_weighted_trade_net_return`: gross + funding - costs.

This is a simple summed smoke score. It is not a portfolio equity curve, Sharpe ratio, or production backtest.

In [5]:
notes = (screen_result.result_dir / "notes.md").read_text()
display(Markdown(notes))

# Run Complete

strategy_id: crypto_perp_autoresearch_ensemble_smoke
mode: screen
status: screened
return_scope: price-and-funding; supplied funding events are included when they fall inside engine-held intervals.
interpretation: runner screen evidence only; not validation pass, market robustness, or promotion evidence.


## Inspect Signals

`signals.csv` is the engine-facing view of strategy decisions. The `metadata` column keeps the audit details: vote counts, indicator values, dynamic threshold, target weight, and explicit notes about unsupported upstream stateful features.

In [6]:
def read_signals(result_dir: Path) -> list[dict[str, object]]:
    with (result_dir / "signals.csv").open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        row["metadata"] = json.loads(row["metadata"])
        row["weight"] = float(row["weight"])
        row["max_hold_bars"] = int(row["max_hold_bars"])
        row["trailing_stop_bps"] = float(row["trailing_stop_bps"])
    return rows


signals = read_signals(screen_result.result_dir)
signals[:3]

[{'symbol': 'BTC-PERP',
  'decision_time': '2024-01-03T00:01:00+00:00',
  'as_of_time': '2024-01-03T00:00:00+00:00',
  'side': 'short',
  'weight': 0.02666666666666667,
  'max_hold_bars': 720,
  'trailing_stop_bps': 585.0554970830356,
  'metadata': {'atr_bps': 106.3737267423701,
   'bb_width_percentile': 5.0,
   'dynamic_threshold': 0.008154585359472189,
   'dynamic_threshold_bps': 81.54585359472189,
   'effective_target_weight': 0.02666666666666667,
   'ema_fast': 45116.6947146714,
   'ema_slow': 44880.5963840135,
   'latest_funding_rate': 0.00021159,
   'long_votes': 2,
   'macd_histogram': -143.53424476871317,
   'min_votes': 4,
   'momentum_12h_bps': -97.75725593667461,
   'momentum_6h_bps': 4.554047659774962,
   'portfolio_budget_pct': 0.08,
   'realized_vol': 0.005386463398680472,
   'rsi': 40.2558982323793,
   'same_symbol_overlap_policy': 'suppress_until_assumed_hold_window_end',
   'short_votes': 4,
   'signal_family': 'crypto_perp_autoresearch_ensemble',
   'signal_flip_suppo

## Quant-Trading Sanity Checks

These checks are not a substitute for validation, but they catch two common research mistakes:

- using information after the decision timestamp;
- opening multiple overlapping trades on the same symbol when the engine treats signals independently.

In [7]:
decision_records = [
    json.loads(line)
    for line in (screen_result.result_dir / "decision_records.jsonl").read_text().splitlines()
    if line.strip()
]

causal_violations = [
    record
    for record in decision_records
    if parse_ts(record["as_of_time"]) > parse_ts(record["decision_time"])
]

evidence = read_json(screen_result.result_dir / "evidence.json")
trades = evidence["screening_result"]["trades"]


def same_symbol_overlaps(trades: list[dict]) -> list[tuple[str, str, str]]:
    overlaps = []
    by_symbol: dict[str, list[dict]] = {}
    for trade in trades:
        by_symbol.setdefault(trade["symbol"], []).append(trade)
    for symbol, symbol_trades in by_symbol.items():
        ordered = sorted(symbol_trades, key=lambda item: parse_ts(item["entry_time"]))
        for previous, current in zip(ordered, ordered[1:]):
            if parse_ts(previous["exit_time"]) > parse_ts(current["entry_time"]):
                overlaps.append((symbol, previous["exit_time"], current["entry_time"]))
    return overlaps


overlaps = same_symbol_overlaps(trades)
unique_weights = sorted({row["weight"] for row in signals})

assert not causal_violations
assert not overlaps

{
    "decision_count": len(decision_records),
    "causal_violations": len(causal_violations),
    "same_symbol_overlaps": overlaps,
    "unique_signal_weights": unique_weights,
}

{'decision_count': 26,
 'causal_violations': 0,
 'same_symbol_overlaps': [],
 'unique_signal_weights': [0.02666666666666667]}

Expected interpretation:

- `causal_violations = 0`: every decision uses an `as_of_time` at or before `decision_time`.
- `same_symbol_overlaps = []`: the strategy did not ask the smoke engine to hold overlapping trades in the same symbol.
- `unique_signal_weights` should show about `0.0266667` for BTC/ETH/SOL, because the 8% portfolio budget is split across three symbols.

## Validation Mode

The committed config is `screen` mode. For the notebook, create a generated validation config under ignored `results/notebook_configs/` by changing only `mode = "screen"` to `mode = "validate"`.

Runner validation applies default smoke gates:

- inputs are valid;
- at least one trade exists;
- summed weighted gross return is positive;
- summed weighted net return is positive.

In [8]:
validation_config = REPO_ROOT / "results" / "notebook_configs" / "crypto_perp_autoresearch_ensemble_validate.toml"
validation_config.parent.mkdir(parents=True, exist_ok=True)
validation_config.write_text(SCREEN_CONFIG.read_text().replace('mode = "screen"', 'mode = "validate"'))

validate_result = run_config(validation_config, repo_root=REPO_ROOT)
assert validate_result.success, validate_result.message
validate_result.result_dir

PosixPath('/Users/Season_Yang/Personal/quant_strategies/results/2026-05-27T125151Z-crypto_perp_autoresearch_ensemble_smoke')

In [9]:
validate_summary = read_json(validate_result.result_dir / "summary.json")
validate_summary["engine"]

{'gates': [{'detail': 'screening completed',
   'name': 'valid_inputs',
   'passed': True},
  {'detail': '26 >= 1', 'name': 'min_trades', 'passed': True},
  {'detail': 'sum_weighted_trade_gross_return=0.00331541964205',
   'name': 'positive_gross',
   'passed': True},
  {'detail': 'sum_weighted_trade_net_return=0.00335141297539',
   'name': 'positive_net',
   'passed': True}],
 'passed': True,
 'smoke_score': {'sum_weighted_trade_cost_return': 0.0,
  'sum_weighted_trade_funding_return': 3.599333333333333e-05,
  'sum_weighted_trade_gross_return': 0.0033154196420537695,
  'sum_weighted_trade_net_return': 0.003351412975387103},
 'trade_count': 26}

### How To Read Validation Output

- `passed`: whether all smoke validation gates passed.
- `gates`: individual pass/fail checks and their details.
- `trade_count` and `smoke_score`: same engine metrics as screen mode.
- `assessment_status = smoke_passed`: the runner completed validation gates, but this is still smoke evidence.
- `promotion_eligible = false`, `paper_trade_eligible = false`, `requires_manual_approval = true`: validation output is not enough to promote or trade the strategy.

Treat a pass here as: “the strategy produced causally valid, engine-fillable smoke evidence over this small configured window.” Do not treat it as: “the strategy is market validated.”

In [10]:
display(Markdown((validate_result.result_dir / "notes.md").read_text()))

# Run Complete

strategy_id: crypto_perp_autoresearch_ensemble_smoke
mode: validate
status: passed
return_scope: price-and-funding; supplied funding events are included when they fall inside engine-held intervals.
interpretation: runner smoke evidence only; not market robustness or promotion evidence.
